# generalized-ipu 기본 사용법

표본 미시 자료의 각 개체에 가중치를 부여하여, 가중합이 외부에서 주어진 여러 집계 통계를 동시에 재현하도록 조정하는 전 과정을 다룬다. 라이브러리가 다루는 여섯 가지 문제를 한 예제에서 모두 사용하고, 각각을 실행 가능한 검증으로 확인한다.

| 문제 | 이 노트북에서의 사용 | 검증 절 |
| --- | --- | --- |
| $N$계층 구조 | 지역 - 거처 - 가구 - 개인의 4계층. 기본 단위는 가구 | 2, 4 |
| 구간 목표 | 표본조사에서 얻은 연령×성별 표에 오차 한계를 구간으로 부여 | 5, 7 |
| $N$차원 교차표 | 연령(3) × 성별(2), 연령(3) × 면허(2)의 결합 분포 | 4, 7 |
| 범주 위계 | 연령×성별 소범주를 연령 대범주로 집계 | 4 |
| 주변표의 결측 | 2인·3인가구 수가 미공표인 상황에서 목표 구간을 연역 | 9 |
| 상위 계층 개체 수 | 다가구주택 구조에서 거처 수를 분수 귀속으로 계상 | 4, 8 |

여기에 구조적 영(현실에 존재할 수 없는 조합)과 표본 영(모집단에는 있으나 표본에 없는 조합)의 구별 처리를 함께 보인다.

### 검증 방침

각 절에서 확인한 사실을 `check()` 로 기록한다. 조건이 거짓이면 그 자리에서 `AssertionError` 가 발생하므로, 노트북이 끝까지 실행되었다는 것 자체가 모든 검증을 통과했다는 뜻이다. 마지막 절에서 기록을 표로 모아 확인한다.

용어와 기호는 [`docs/01_용어와_정의.md`](../docs/01_용어와_정의.md)를 따른다. 처음 보는 용어가 나오면 그 문서를 참조한다.

## 0. 준비

라이브러리와 표준 도구를 불러온다. 저장소 최상위에서 `pip install -e .` 를 먼저 실행해야 한다.

In [1]:
import numpy as np
import pandas as pd

import generalized_ipu as gipu
from generalized_ipu.constraints import compute_update_ratios

print("generalized-ipu", gipu.__version__)

CHECKS = []


def check(requirement, description, condition):
    # 검증 결과를 기록한다. 조건이 거짓이면 즉시 예외를 발생시킨다.
    passed = bool(condition)
    CHECKS.append({"문제": requirement, "검증 내용": description, "통과": passed})
    if not passed:
        raise AssertionError(f"검증 실패: {description}")
    return passed

generalized-ipu 0.1.0.dev0


## 1. 예제 자료 생성

실제 상황을 모사하기 위해 **모집단**을 먼저 만들고, 그중 일부를 **표본**으로 추출한다. 목표값은 모집단에서 계산하고, 조정은 표본에만 수행한다. 이렇게 하면 목표가 실현 가능함이 보장되므로 알고리즘의 동작을 명확히 관찰할 수 있다.

계층은 네 개이다.

- **지역** 2곳
- **거처** 1,600호. 대부분은 한 가구가 살지만 일부는 두세 가구가 함께 사는 **다가구주택**이다.
- **가구** 약 2,000호. 각 가구는 소속 거처와 보유 차량 수를 가진다.
- **개인** 약 5,300명. 각 개인은 소속 가구, 연령대, 성별, 운전면허 보유 여부를 가진다.

거처 계층을 따로 두는 이유는 "거처 1호에 $n$개 가구"라는 구조를 제약으로 표현하기 위해서이다. 가구를 기본 단위로 두면 거처는 상위 계층이 되며, 상위 계층 개체의 **수**를 세는 일은 상위 계층의 **값**을 내리는 일과 다르다. 이 구별은 4절과 8절에서 다룬다.

운전면허는 연령대가 `child` 인 개인에게 부여하지 않는다. 이 조건이 뒤에서 구조적 영으로 처리된다.

In [2]:
rng = np.random.default_rng(20260906)

N_DWELLINGS = 1600
AGE_GROUPS = ["child", "adult", "senior"]
SEXES = ["M", "F"]
HH_CLASSES = ["1", "2", "3+"]          # 거처 1호에 사는 가구 수의 계급
SIZE_CLASSES = ["1", "2", "3", "4+"]   # 가구원 수의 계급

regions = pd.DataFrame({"region_id": ["r1", "r2"], "urban": [1.0, 0.0]})

# 거처: 소속 지역과 그 거처에 사는 가구 수
households_per_dwelling = rng.choice([1, 2, 3], size=N_DWELLINGS, p=[0.80, 0.15, 0.05])
dwellings = pd.DataFrame(
    {
        "dwelling_id": [f"d{i:05d}" for i in range(N_DWELLINGS)],
        "region_id": rng.choice(["r1", "r2"], size=N_DWELLINGS, p=[0.6, 0.4]),
        "hh_class": gipu.AggregationMapper.size_class_labels(
            households_per_dwelling, boundaries=(1, 2, 3)
        ).to_numpy(),
    }
)

# 가구: 거처마다 그 거처의 가구 수만큼 생성한다.
dwelling_ids = np.repeat(dwellings["dwelling_id"].to_numpy(), households_per_dwelling)
n_households = dwelling_ids.size
households = pd.DataFrame(
    {
        "hh_id": [f"h{i:05d}" for i in range(n_households)],
        "dwelling_id": dwelling_ids,
        "car": rng.choice([0, 1, 2], size=n_households, p=[0.25, 0.55, 0.20]).astype(float),
    }
)

# 개인: 가구마다 가구원 수를 뽑고 각각의 속성을 생성한다.
sizes = rng.choice([1, 2, 3, 4, 5], size=n_households, p=[0.20, 0.30, 0.25, 0.15, 0.10])
hh_ids = np.repeat(households["hh_id"].to_numpy(), sizes)
n_persons = hh_ids.size

age_group = rng.choice(AGE_GROUPS, size=n_persons, p=[0.20, 0.60, 0.20])
sex = rng.choice(SEXES, size=n_persons, p=[0.49, 0.51])

# 운전면허: child 는 논리적으로 보유할 수 없다.
license_prob = np.where(age_group == "child", 0.0, np.where(age_group == "adult", 0.85, 0.55))
has_license = (rng.random(n_persons) < license_prob).astype(int)

persons = pd.DataFrame(
    {
        "person_id": [f"p{i:06d}" for i in range(n_persons)],
        "hh_id": hh_ids,
        "age_group": age_group,
        "sex": sex,
        "license": has_license,
    }
)

print(f"거처 {len(dwellings):,}호, 가구 {len(households):,}호, 개인 {len(persons):,}명")
print(f"거처당 가구 수 분포: {dict(dwellings['hh_class'].value_counts().sort_index())}")
dwellings.head()

거처 1,600호, 가구 2,014호, 개인 5,342명
거처당 가구 수 분포: {'1': np.int64(1274), '2': np.int64(238), '3+': np.int64(88)}


,dwelling_id,region_id,hh_class
0,d00000,r1,2
1,d00001,r1,1
2,d00002,r1,1
3,d00003,r2,1
4,d00004,r2,1


### 표본 추출

**거처** 240호를 뽑아 표본으로 삼는다. 거처를 뽑으면 그에 속한 가구가, 가구를 뽑으면 그에 속한 개인이 함께 따라온다.

기본 단위는 가구인데 왜 거처 단위로 뽑는가. 다가구주택의 가구 중 일부만 표본에 들어오면, 표본에서 관찰되는 "거처당 가구 수"가 모집단의 값과 달라진다. 그러면 분수 귀속의 분모 $m$ 이 틀린 값이 되어 거처 수 추정이 어긋난다. 상위 계층 개체 수를 제약으로 쓰려면 추출 단위가 그 상위 계층이거나, 최소한 상위 개체가 통째로 들어오고 나가야 한다.

In [3]:
N_SAMPLE_DWELLINGS = 240

sample_dwelling_ids = rng.choice(
    dwellings["dwelling_id"].to_numpy(), size=N_SAMPLE_DWELLINGS, replace=False
)
sample_dwellings = (
    dwellings[dwellings["dwelling_id"].isin(sample_dwelling_ids)].reset_index(drop=True)
)
sample_households = (
    households[households["dwelling_id"].isin(sample_dwelling_ids)].reset_index(drop=True)
)
sample_persons = (
    persons[persons["hh_id"].isin(set(sample_households["hh_id"]))].reset_index(drop=True)
)

print(f"표본 거처 {len(sample_dwellings):,}호, 가구 {len(sample_households):,}호, "
      f"개인 {len(sample_persons):,}명")
print(f"거처 추출률 {len(sample_dwellings) / len(dwellings):.1%}")

check(
    "N계층 구조",
    "표본은 거처 단위로 추출되어 다가구주택의 가구가 통째로 포함된다",
    sample_households["dwelling_id"].nunique() == len(sample_dwellings),
)

표본 거처 240호, 가구 309호, 개인 859명
거처 추출률 15.0%


True

## 2. 계층 트리 구성

`HierarchyTree` 는 계층들의 부모-자식 관계를 관리한다. 각 계층은 **주키**로 자기 행을 식별하고, **부모키**로 상위 계층 행을 참조한다.

`base_level` 로 지정한 계층이 **기본 계층**이며, 그 행 하나하나가 가중치를 직접 부여받는 **기본 단위**가 된다. 여기서는 가구를 기본 단위로 삼는다. 거처 제약과 개인 제약은 모두 가구 좌표계로 집계되어 처리된다.

계층이 넷이므로 지역은 기본 계층보다 두 단계 위에 있다. 라이브러리는 부모키를 따라 경로를 거슬러 올라가므로 몇 단계 떨어져 있든 동일하게 처리한다.

트리를 만든 뒤 **가구원 수**를 세어 계급 열로 파생시킨다. $n$인가구 제약은 이 열의 교차표로 표현된다. 가구원 수는 원본 자료에 없는 값이며, 계층 구조에서 유도된다.

In [4]:
def build_tree(dwelling_frame, household_frame, person_frame):
    tree = gipu.HierarchyTree(base_level="household")
    tree.add_level("region", regions, primary_key="region_id")
    tree.add_level(
        "dwelling",
        dwelling_frame,
        primary_key="dwelling_id",
        parent_key="region_id",
        parent_level="region",
    )
    tree.add_level(
        "household",
        household_frame,
        primary_key="hh_id",
        parent_key="dwelling_id",
        parent_level="dwelling",
    )
    tree.add_level(
        "person",
        person_frame,
        primary_key="person_id",
        parent_key="hh_id",
        parent_level="household",
    )
    tree.validate_integrity()

    # n인가구 계급: 하위 계층 개체 수를 세어 계급 이름으로 변환한다.
    counts = gipu.AggregationMapper.descendant_count_to_base(tree, "person")
    tree.nodes["household"].data["size_class"] = (
        gipu.AggregationMapper.size_class_labels(counts, boundaries=(1, 2, 3, 4)).to_numpy()
    )
    return tree


sample_tree = build_tree(sample_dwellings, sample_households, sample_persons)
population_tree = build_tree(dwellings, households, persons)

print("기본 계층과의 관계")
for level in ["region", "dwelling", "household", "person"]:
    print(f"  {level:10s} -> {sample_tree.relation_to_base(level)}")

print()
print("가구원 수 계급 분포 (표본)")
print(sample_tree.nodes["household"].data["size_class"].value_counts().sort_index())

기본 계층과의 관계
  region     -> ancestor
  dwelling   -> ancestor
  household  -> base
  person     -> descendant

가구원 수 계급 분포 (표본)
size_class
1     54
2     84
3     85
4+    86
Name: count, dtype: int64


무결성 조건이 왜 필요한지는 위반 사례로 확인하는 편이 빠르다. 부모키가 상위 계층에 없는 값을 가리키면, 검증 없이는 그 행이 집계에서 조용히 누락되어 과소 집계로 이어진다.

In [5]:
broken_persons = sample_persons.copy()
broken_persons.loc[0, "hh_id"] = "h99999"  # 존재하지 않는 가구

try:
    build_tree(sample_dwellings, sample_households, broken_persons)
    raised = False
except ValueError as error:
    raised = True
    print("검증에서 걸러짐:", error)

check("N계층 구조", "참조 무결성 위반을 반복 이전에 예외로 걸러낸다", raised)

검증에서 걸러짐: 'person' 계층의 부모키 'hh_id' 중 1건이 'household' 계층의 주키와 대응하지 않고, 0건이 결측입니다.


True

## 3. 제약 열의 정의

**제약 열**은 목표값 하나가 대응되는 최소 제약 단위이다. 1차원 주변표의 범주 하나, $N$차원 교차표의 셀 하나, 대범주 하나가 각각 하나의 제약 열이 된다.

이 예제에서는 여섯 개의 블록을 쓴다.

| 블록 | 종류 | 계층 | 제약 열 | 목표의 성격 |
| --- | --- | --- | --- | --- |
| `hh_car` | `ColumnBlock` | 가구 | 차량 대수 1개 | 등록 통계이므로 고정 스칼라 |
| `person_age_sex` | `CrossTabBlock` | 개인 | 연령 3 × 성별 2 = 6개 | 표본조사이므로 ±8% 구간 |
| `person_age` | `CoarseBlock` | 개인 | 연령 대범주 3개 | 인구총조사이므로 고정 스칼라 |
| `person_license` | `CrossTabBlock` | 개인 | 연령 3 × 면허 2 = 6개 중 유효 5개 | 고정 스칼라 |
| `dwelling_class` | `ShareBlock` | 거처 | 가구 수 계급 3개 | 주택총조사이므로 고정 스칼라 |
| `hh_size` | `CrossTabBlock` | 가구 | 가구원 수 계급 4개 | 9절에서 일부를 결측으로 바꾼다 |

`person_age` 는 `person_age_sex` 의 소범주를 연령 기준으로 묶은 **대범주** 블록이다. 같은 속성이 표마다 다른 세분화 수준으로 주어지는 상황을 이렇게 처리한다.

`dwelling_class` 는 상위 계층인 거처의 **개수**를 세는 블록이다. 다른 블록과 달리 성분이 0과 1이 아니라 $1/m$ 의 분수이며, 그 이유는 4절에서 확인한다.

In [6]:
AGE_SEX_DIMS = {"age_group": AGE_GROUPS, "sex": SEXES}
AGE_LICENSE_DIMS = {"age_group": AGE_GROUPS, "license": [0, 1]}
DWELLING_DIMS = {"hh_class": HH_CLASSES}
SIZE_DIMS = {"size_class": SIZE_CLASSES}

# 연령×성별 소범주를 연령 대범주로 묶는 위계
age_hierarchy = gipu.CategoryHierarchy("age_group")
for age in AGE_GROUPS:
    age_hierarchy.add_mapping(
        f"age={age}", [f"age_group={age}|sex={s}" for s in SEXES]
    )

age_hierarchy.mapping

{'age=child': ['age_group=child|sex=M', 'age_group=child|sex=F'],
 'age=adult': ['age_group=adult|sex=M', 'age_group=adult|sex=F'],
 'age=senior': ['age_group=senior|sex=M', 'age_group=senior|sex=F']}

### 구조적 영의 지정

`child` 이면서 운전면허를 보유한 조합은 현실에 존재할 수 없다. 이런 조합에 대응하는 제약 열을 **구조적 영**이라 하며, 속성 행렬을 구축할 때 해당 열의 색인을 아예 할당하지 않는다. 이를 **색인 원천 제외**라 한다.

`LogicalRuleParser` 는 불가능한 조합을 기술하는 규칙을 받아 **유효 마스크**를 산출한다. 마스크는 **유효한 쪽이 참**이고, 길이는 기본 단위의 수가 아니라 제약 열의 수와 같다. 규칙 평가의 대상은 표본 자료의 행이 아니라 제약 열이 나타내는 범주 조합이다.

In [7]:
license_columns = gipu.TensorTarget(
    "person_license", "person", AGE_LICENSE_DIMS, np.zeros((3, 2))
)
category_frame = license_columns.category_frame()

parser = gipu.LogicalRuleParser(["age_group == 'child' and license == 1"])
license_validity = parser.build_validity_mask(category_frame)

pd.concat(
    [category_frame, pd.Series(license_validity, name="유효")], axis=1
)

,age_group,license,유효
0,child,0,True
1,child,1,False
2,adult,0,True
3,adult,1,True
4,senior,0,True
5,senior,1,True


In [8]:
print(parser.describe(category_frame).to_string(index=False))

check(
    "구조적 영",
    "규칙이 지정한 조합 1개만 유효 마스크에서 거짓이 된다",
    int((~license_validity).sum()) == 1,
)

                                 rule  n_columns_excluded
age_group == 'child' and license == 1                   1


True

## 4. 속성 행렬 구축

**속성 행렬** $A$ 는 $n \times K$ 희소 행렬이며, 성분 $a_{ik}$ 는 기본 단위 $i$ 가 가중치 1을 가질 때 제약 열 $k$ 에 기여하는 수량이다. 기본 단위가 가구이고 제약 열이 "성인 남성 수"이면, $a_{ik}$ 는 그 가구에 속한 성인 남성의 인원수이다.

$N$계층, $N$차원 교차표, 범주 위계, 상위 계층 개체 수라는 확장 요건은 모두 "제약 열이 늘어난다"는 하나의 문제로 환원된다. 따라서 반복 루프는 제약의 출처를 알 필요가 없고, $\mathbf{S} = A^{\top}\mathbf{w}$ 라는 단일 연산만 수행한다.

In [9]:
BLOCKS = [
    gipu.ColumnBlock(name="hh_car", level="household", cols=["car"]),
    gipu.CrossTabBlock(name="person_age_sex", level="person", dims=AGE_SEX_DIMS),
    gipu.CoarseBlock(name="person_age", source="person_age_sex", hierarchy=age_hierarchy),
    gipu.CrossTabBlock(name="person_license", level="person", dims=AGE_LICENSE_DIMS),
    gipu.ShareBlock(name="dwelling_class", level="dwelling", dims=DWELLING_DIMS),
    gipu.CrossTabBlock(name="hh_size", level="household", dims=SIZE_DIMS),
]
VALIDITY_MASKS = {"person_license": license_validity}


def build_attribute_matrix(tree):
    return gipu.UnifiedSparseMatrixBuilder(tree).build(BLOCKS, validity_masks=VALIDITY_MASKS)


attribute = build_attribute_matrix(sample_tree)

print(f"기본 단위 {attribute.n_units:,}개, 제약 열 {attribute.n_columns}개")
print(f"비영 성분 {attribute.nnz:,}개, "
      f"밀도 {attribute.nnz / (attribute.n_units * attribute.n_columns):.1%}")
attribute.column_frame()

기본 단위 309개, 제약 열 22개
비영 성분 2,691개, 밀도 39.6%


,block,column
0,hh_car,household.car
1,person_age_sex,age_group=child|sex=M
2,person_age_sex,age_group=child|sex=F
3,person_age_sex,age_group=adult|sex=M
4,person_age_sex,age_group=adult|sex=F
5,person_age_sex,age_group=senior|sex=M
6,person_age_sex,age_group=senior|sex=F
7,person_age,age=child
8,person_age,age=adult
9,person_age,age=senior


`person_license` 블록의 제약 열이 6개가 아니라 5개이다. `age_group=child|license=1` 이 구조적 영으로 판정되어 색인이 할당되지 않았다.

대범주 블록이 소범주 블록의 합과 일치하는지도 확인해 둔다. $A_{coarse} = A_{fine} M^{\top}$ 관계가 성립한다.

In [10]:
fine = attribute.block("person_age_sex").toarray()
coarse = attribute.block("person_age").toarray()
mapping = gipu.MappingMatrix.build_binary_matrix(
    age_hierarchy, attribute.column_names[attribute.block_slices["person_age_sex"]]
)

check(
    "구조적 영",
    "구조적 영 제약 열은 속성 행렬에 색인이 할당되지 않는다",
    attribute.block("person_license").shape[1] == 5,
)
check(
    "범주 위계",
    "대범주 블록이 소범주 블록과 매핑 행렬의 곱과 정확히 일치한다",
    np.allclose(coarse, fine @ mapping.toarray().T),
)

pd.DataFrame(
    {
        "대범주": age_hierarchy.coarse_categories,
        "소범주 합": (fine @ mapping.toarray().T).sum(axis=0),
        "대범주 열": coarse.sum(axis=0),
    }
)

,대범주,소범주 합,대범주 열
0,age=child,186.0,186.0
1,age=adult,481.0,481.0
2,age=senior,192.0,192.0


### 분수 귀속의 확인

상위 계층인 거처를 다루는 방식은 두 가지이며, 둘은 서로 다른 것을 계산한다.

| 방식 | 함수 | 기본 단위가 받는 값 | 가중합의 의미 |
| --- | --- | --- | --- |
| 방송 | `aggregate_to_base` | 거처의 값을 그대로 복제 | 기본 단위 수만큼 중복 계상 |
| 분수 귀속 | `ancestor_share_to_base` | 거처의 값을 $1/m$ 로 나눔 | 거처 수 |

거처 1호에 가구 $m$ 개가 살 때, 값이 1인 열을 방송하면 그 열의 가중합은 거처 수가 아니라 **가구 수**가 된다. `ShareBlock` 은 각 가구에 $1/m$ 을 부여하여 거처 하나가 정확히 한 번만 세어지게 한다.

In [11]:
# 값이 1인 열을 방송하면 무엇이 세어지는지 확인한다.
broadcast_tree = build_tree(sample_dwellings, sample_households, sample_persons)
broadcast_tree.nodes["dwelling"].data["unit"] = 1.0
broadcast = gipu.AggregationMapper.aggregate_to_base(broadcast_tree, "dwelling", ["unit"])

share_block = attribute.block("dwelling_class")
unit_weights = np.ones(attribute.n_units)

print(f"방송한 열의 가중합    {float(broadcast['unit'].sum()):8.1f}  <- 표본 가구 수 {len(sample_households):,}")
print(f"분수 귀속 열의 가중합  {float(share_block.T.dot(unit_weights).sum()):8.1f}  <- 표본 거처 수 {len(sample_dwellings):,}")

check(
    "상위 계층 개체 수",
    "방송한 열의 가중합은 거처 수가 아니라 가구 수가 된다",
    np.isclose(float(broadcast["unit"].sum()), len(sample_households)),
)
check(
    "상위 계층 개체 수",
    "분수 귀속 열의 가중합은 표본 거처 수와 일치한다",
    np.isclose(float(share_block.T.dot(unit_weights).sum()), len(sample_dwellings)),
)

# 계급별로도 정확히 일치하는지 확인한다.
observed_by_class = share_block.T.dot(unit_weights)
expected_by_class = (
    sample_dwellings["hh_class"].value_counts().reindex(HH_CLASSES).to_numpy(dtype=float)
)
check(
    "상위 계층 개체 수",
    "가구 수 계급별 거처 수가 표본의 실제 분포와 일치한다",
    np.allclose(observed_by_class, expected_by_class),
)

pd.DataFrame(
    {
        "계급": HH_CLASSES,
        "분수 귀속 가중합": observed_by_class,
        "표본 거처 수": expected_by_class,
    }
)

방송한 열의 가중합       309.0  <- 표본 가구 수 309
분수 귀속 열의 가중합     240.0  <- 표본 거처 수 240


,계급,분수 귀속 가중합,표본 거처 수
0,1,184.0,184.0
1,2,43.0,43.0
2,3+,13.0,13.0


$n$인가구 블록도 같은 방식으로 확인한다. 가구원 수는 원본 자료에 없고 계층 구조에서 유도한 값이므로, 유도가 맞았는지 개인 자료로 직접 세어 대조한다.

In [12]:
size_observed = attribute.block("hh_size").T.dot(unit_weights)
size_expected = (
    sample_tree.nodes["household"]["size_class"].value_counts().reindex(SIZE_CLASSES).to_numpy(float)
    if False
    else sample_tree.nodes["household"].data["size_class"]
    .value_counts()
    .reindex(SIZE_CLASSES)
    .to_numpy(dtype=float)
)
direct_counts = sample_persons.groupby("hh_id").size()

check(
    "N계층 구조",
    "계층에서 유도한 가구원 수가 개인 자료를 직접 센 값과 일치한다",
    np.array_equal(
        gipu.AggregationMapper.descendant_count_to_base(sample_tree, "person").to_numpy(),
        direct_counts.reindex(sample_households["hh_id"]).to_numpy(dtype=float),
    ),
)
check(
    "N차원 교차표",
    "가구원 수 계급별 가구 수가 표본의 실제 분포와 일치한다",
    np.allclose(size_observed, size_expected),
)

pd.DataFrame({"계급": SIZE_CLASSES, "가중합": size_observed, "표본 가구 수": size_expected})

,계급,가중합,표본 가구 수
0,1,54.0,54.0
1,2,84.0,84.0
2,3,85.0,85.0
3,4+,86.0,86.0


## 5. 목표값의 정의

목표는 모두 **구간** $[L_k, U_k]$ 로 표현한다. 고정 스칼라 목표는 $L_k = U_k$ 인 퇴화 구간으로 취급하므로, 라이브러리 내부에 스칼라 목표라는 별도의 자료형은 존재하지 않는다.

모집단에 동일한 블록 구성을 적용하고 가중치를 모두 1로 두면, 가중합이 곧 참값이 된다. 이 값을 목표로 사용한다.

In [13]:
population_attribute = build_attribute_matrix(population_tree)
population_totals = population_attribute.matrix.T.dot(np.ones(population_attribute.n_units))

targets = pd.DataFrame(
    {"column": population_attribute.column_names, "population_total": population_totals}
)
targets.insert(0, "block", attribute.column_frame()["block"])
targets

,block,column,population_total
0,hh_car,household.car,1945.0
1,person_age_sex,age_group=child|sex=M,567.0
2,person_age_sex,age_group=child|sex=F,517.0
3,person_age_sex,age_group=adult|sex=M,1558.0
4,person_age_sex,age_group=adult|sex=F,1591.0
5,person_age_sex,age_group=senior|sex=M,531.0
6,person_age_sex,age_group=senior|sex=F,578.0
7,person_age,age=child,1084.0
8,person_age,age=adult,3149.0
9,person_age,age=senior,1109.0


### 제약 등록

`ConstraintRegistry` 는 제약 객체를 등록 순서대로 보관하고, 각 제약이 점유하는 제약 열 구간을 관리한다. 등록 순서는 속성 행렬의 블록 순서와 일치해야 한다.

- `hh_car`, `person_age`, `person_license`, `dwelling_class`, `hh_size` 는 고정 스칼라 목표를 받는다.
- `person_age_sex` 는 표본조사 결과이므로 ±8% 구간을 받는다.
- `person_license` 는 구조적 영이 제외된 5개 열에 대해서만 목표를 받는다.

In [14]:
def totals_of(block_name):
    span = population_attribute.block_slices[block_name]
    return population_totals[span]


def names_of(block_name):
    return attribute.column_names[attribute.block_slices[block_name]]


BOUND_MARGIN = 0.08
age_sex_totals = totals_of("person_age_sex")


def build_registry(hh_size_lower=None, hh_size_upper=None):
    # 블록 순서대로 제약을 등록한다. 가구 규모 목표만 바꿔 끼울 수 있게 둔다.
    built = gipu.ConstraintRegistry()
    built.register(
        gipu.BoundChecker("hh_car", "household", {"car": float(totals_of("hh_car")[0])})
    )
    built.register(
        gipu.BoundChecker.from_arrays(
            "person_age_sex", "person", names_of("person_age_sex"),
            age_sex_totals * (1 - BOUND_MARGIN), age_sex_totals * (1 + BOUND_MARGIN),
        )
    )
    built.register(
        gipu.BoundChecker.from_arrays(
            "person_age", "person", names_of("person_age"), totals_of("person_age")
        )
    )
    built.register(
        gipu.BoundChecker.from_arrays(
            "person_license", "person", names_of("person_license"), totals_of("person_license")
        )
    )
    built.register(
        gipu.BoundChecker.from_arrays(
            "dwelling_class", "dwelling", names_of("dwelling_class"), totals_of("dwelling_class")
        )
    )
    lower = totals_of("hh_size") if hh_size_lower is None else hh_size_lower
    upper = totals_of("hh_size") if hh_size_upper is None else hh_size_upper
    built.register(
        gipu.BoundChecker.from_arrays("hh_size", "household", names_of("hh_size"), lower, upper)
    )
    built.validate_against(attribute.n_columns)
    return built


registry = build_registry()
lower, upper = registry.bounds()

pd.DataFrame(
    {
        "제약": registry.column_owners,
        "제약 열": attribute.column_names,
        "하한": lower,
        "상한": upper,
        "구간 여부": lower != upper,
    }
)

,제약,제약 열,하한,상한,구간 여부
0,hh_car,household.car,1945.00,1945.00,False
1,person_age_sex,age_group=child|sex=M,521.64,612.36,True
2,person_age_sex,age_group=child|sex=F,475.64,558.36,True
3,person_age_sex,age_group=adult|sex=M,1433.36,1682.64,True
4,person_age_sex,age_group=adult|sex=F,1463.72,1718.28,True
5,person_age_sex,age_group=senior|sex=M,488.52,573.48,True
6,person_age_sex,age_group=senior|sex=F,531.76,624.24,True
7,person_age,age=child,1084.00,1084.00,False
8,person_age,age=adult,3149.00,3149.00,False
9,person_age,age=senior,1109.00,1109.00,False


## 6. 반복 수렴

`GeneralizedIPUEngine` 이 반복을 주도한다. 한 회차는 다음 순서로 진행한다.

1. 갱신 비율 $r_k$ 산출. 가중합이 목표 구간 안에 있으면 $r_k = 1$ 이므로 가중치를 건드리지 않는다.
2. 동시 갱신식으로 새 가중치 후보 계산
3. 배율 제한과 가중치 하한 적용
4. 가중합 재계산 및 수렴 판정
5. 이력 기록

$$w_i^{(t+1)} = w_i^{(t)}\left(1 + \eta \cdot \frac{\sum_{k}(r_k^{(t)} - 1)\,a_{ik}}{\sum_{k} a_{ik}}\right)$$

$\eta$ 는 **완화 계수**로 갱신의 보폭을 조절한다. 값이 작을수록 진동이 줄고 수렴이 느려진다.

초기 가중치는 추출률의 역수로 둔다. 별도로 지정하지 않으면 모두 1에서 출발한다.

In [15]:
expansion_factor = len(dwellings) / len(sample_dwellings)
initial_weights = np.full(attribute.n_units, expansion_factor)

engine = gipu.GeneralizedIPUEngine.from_registry(
    attribute.matrix, registry, relative_gap=0.005, max_iterations=300, eta=1.0
)
result = engine.fit(initial_weights)

print(result)
print()
print(f"종료 사유      : {result.termination_reason}")
print(f"소요 회차      : {result.n_iterations}")
print(f"최대 상대 격차 : {result.report.max_relative_gap:.6f}")
print(f"미충족 제약 열 : {result.report.n_violations} / {attribute.n_columns}")

check("구간 목표", "여섯 블록의 제약을 동시에 부과한 반복이 수렴한다", result.converged)

IPUResult(termination_reason='converged', n_iterations=84, max_relative_gap=0.0049554, n_violations=0)

종료 사유      : converged
소요 회차      : 84
최대 상대 격차 : 0.004955
미충족 제약 열 : 0 / 22


True

**수렴**과 **최대 반복 도달**은 서로 다른 종료 사유이다. `termination_reason` 으로 구분한다. 최대 반복에 도달했다는 것은 알고리즘이 실패했다는 뜻이 아니라, 주어진 제약이 실현 불가능할 수 있다는 신호로 먼저 해석해야 한다.

### 수렴 궤적

`ConvergenceTracker` 가 회차마다 지표를 기록한다. 회차 0은 초기 가중치에 대한 판정이므로, 기록의 수는 항상 회차 수보다 하나 많다.

In [16]:
history = result.tracker.to_frame()
print(f"기록 {len(history)}건 (회차 {result.n_iterations} + 초기 상태)")
print(f"진단 소견: {result.tracker.diagnose()}")

history.head(12)

기록 85건 (회차 84 + 초기 상태)
진단 소견: 감소 중


,iteration,max_relative_gap,max_absolute_diff,violation_rate,n_violations,n_ratio_clipped,n_floor_clipped
0,0,0.253333,171.000000,0.863636,19,0,0
1,1,0.172012,95.585668,0.818182,18,0,0
2,2,0.143418,86.924820,0.772727,17,0,0
3,3,0.128244,79.131269,0.681818,15,0,0
4,4,0.122298,71.340810,0.727273,16,0,0
5,5,0.115885,64.680967,0.727273,16,0,0
6,6,0.109424,58.759276,0.681818,15,0,0
7,7,0.103072,53.433645,0.681818,15,0,0
8,8,0.096906,48.628386,0.636364,14,0,0
9,9,0.090971,44.287987,0.681818,15,0,0


## 7. 적합도 진단

`FitReportGenerator` 는 제약 열별 적합도를 **구간 위반량** $v_k = \max(L_k - S_k,\ S_k - U_k,\ 0)$ 기준으로 정리한다. 가중합이 목표 구간 안에 있으면 위반량이 0이다. 표는 상대 격차의 내림차순으로 정렬되므로 상단이 곧 가장 나쁜 제약 열이다.

판정 열은 두 가지이다. `within_bounds` 는 허용 한계를 고려하지 않고 구간 포함 여부만 엄격히 따지고, `satisfied` 는 엔진의 수렴 판정과 같은 규칙을 적용한다.

In [17]:
report = gipu.FitReportGenerator.generate_report(
    result.weighted_sums,
    *registry.bounds(),
    column_names=attribute.column_names,
    column_owners=registry.column_owners,
    relative_gap=0.005,  # 엔진에 준 허용 한계와 동일하게 둔다
)
report

,constraint,column,lower,upper,estimated,bound_violation,relative_gap,within_bounds,satisfied
0,dwelling_class,hh_class=3+,88.00,88.00,87.563925,0.436075,0.004955,False,True
1,hh_size,size_class=4+,524.00,524.00,521.949947,2.050053,0.003912,False,True
2,hh_size,size_class=3,461.00,461.00,460.145998,0.854002,0.001852,False,True
3,dwelling_class,hh_class=2,238.00,238.00,237.719845,0.280155,0.001177,False,True
4,dwelling_class,hh_class=1,1274.00,1274.00,1272.580810,1.419190,0.001114,False,True
5,hh_size,size_class=2,621.00,621.00,620.497939,0.502061,0.000808,False,True
6,person_license,age_group=senior|license=1,609.00,609.00,609.380238,0.380238,0.000624,False,True
7,person_license,age_group=adult|license=1,2694.00,2694.00,2695.669302,1.669302,0.000620,False,True
8,person_age,age=child,1084.00,1084.00,1084.648262,0.648262,0.000598,False,True
9,person_license,age_group=child|license=0,1084.00,1084.00,1084.648262,0.648262,0.000598,False,True


In [18]:
summary = gipu.FitReportGenerator.summarize_by_constraint(report)

check(
    "구간 목표",
    "허용 한계를 반영한 미충족 제약 열이 하나도 없다",
    int(summary["n_unsatisfied"].sum()) == 0,
)
check(
    "구간 목표",
    "구간 목표를 가진 제약은 구간 안에 들어온 시점부터 조정이 멈춘다",
    float(summary.loc[summary["constraint"] == "person_age_sex", "max_bound_violation"].iloc[0]) == 0.0,
)

summary

,constraint,n_columns,n_outside_bounds,n_unsatisfied,max_relative_gap,max_bound_violation,total_target,total_estimated
0,dwelling_class,3,3,0,0.004955,1.419190,1600.00,1597.864580
1,hh_size,4,4,0,0.003912,2.050053,2014.00,2010.712275
2,person_license,5,5,0,0.000624,1.669302,5342.00,5345.164300
3,person_age,3,3,0,0.000598,1.870160,5342.00,5345.164300
4,hh_car,1,1,0,0.000459,0.893512,1945.00,1944.106488
5,person_age_sex,6,0,0,0.000000,0.000000,4914.64,5345.164300


`n_outside_bounds` 는 목표 구간을 벗어난 제약 열의 수, `n_unsatisfied` 는 허용 한계까지 반영한 미충족 제약 열의 수이다. 스칼라 목표를 가진 제약들은 목표값에 정확히 일치하지 않으므로 `n_outside_bounds` 가 0이 아니지만, 상대 격차가 허용 한계 안에 있으므로 `n_unsatisfied` 는 0이다. 엔진이 수렴으로 판정한 근거가 이것이다.

`person_age_sex` 는 구간 목표이므로 가중합이 구간 안에 들어온 시점부터 조정이 멈춘다. 구간 위반량이 정확히 0인 이유이다.

### 가중치 분포 진단

제약 적합도가 양호하더라도 소수의 가구가 전체 가중치의 대부분을 차지하면, 합성 결과는 표본의 다양성을 잃고 소수 개체의 복제물이 된다. 적합도만으로는 이 상태를 탐지할 수 없으므로 분포를 따로 확인한다.

**유효 표본 수** $(\sum_i w_i)^2 / \sum_i w_i^2$ 가 실제 기본 단위 수에 비해 현저히 작으면, 제약을 완화하거나 배율 제한을 강화하거나 표본을 보강해야 한다는 신호이다.

In [19]:
distribution = gipu.WeightDistributionAnalyzer.analyze(
    result.weights, initial_weights=initial_weights, weight_floor=1e-5
)
distribution.to_series()

n_units                      309.000000
total_weight                2010.712275
minimum                        3.322471
maximum                        9.763253
mean                           6.507159
std                            1.446315
coefficient_of_variation       0.222265
normalized_entropy             0.995741
effective_sample_size        294.453462
top_1_percent_share            0.019275
top_5_percent_share            0.076018
n_at_floor                     0.000000
max_expansion_ratio            1.464488
min_expansion_ratio            0.498371
q01                            3.702972
q25                            5.586120
q50                            6.312708
q75                            7.398725
q99                            9.614542
dtype: float64

In [20]:
concerns = gipu.WeightDistributionAnalyzer.flag_concerns(distribution)
if concerns:
    for item in concerns:
        print("주의:", item)
else:
    print("주의 항목 없음")

print(f"\n유효 표본 수 {distribution.effective_sample_size:.1f} / 기본 단위 {distribution.n_units}")
print(f"가중치 범위 {distribution.minimum:.2f} ~ {distribution.maximum:.2f}")
print(f"초기 가중치 대비 배율 "
      f"{distribution.min_expansion_ratio:.2f} ~ {distribution.max_expansion_ratio:.2f}")

주의 항목 없음

유효 표본 수 294.5 / 기본 단위 309
가중치 범위 3.32 ~ 9.76
초기 가중치 대비 배율 0.50 ~ 1.46


## 8. 다가구주택과 $n$인가구 제약의 재현

조정이 끝난 가중치가 두 제약을 실제로 재현하는지 확인한다. 이 두 제약은 성격이 다르다.

- **거처 수**는 상위 계층 개체의 수이며 분수 귀속으로 계상된다. 가중합이 모집단 거처 수를 재현해야 한다.
- **$n$인가구 수**는 기본 계층 자신의 교차표이며 통상의 지표로 계상된다.

두 제약은 독립이 아니다. 가구 수 계급별 거처 수 $D_m$ 과 총 가구 수 $H$ 사이에는 $\sum_m m D_m = H$ 가, $n$인가구 수 $H_n$ 과 총 가구원 수 $P$ 사이에는 $\sum_n n H_n = P$ 가 성립한다. 조정 결과가 이 항등식을 함께 만족하는지도 확인한다.

In [21]:
dwelling_sums = result.weighted_sums[attribute.block_slices["dwelling_class"]]
size_sums = result.weighted_sums[attribute.block_slices["hh_size"]]

comparison = pd.DataFrame(
    {
        "제약 열": names_of("dwelling_class") + names_of("hh_size"),
        "추정": np.concatenate([dwelling_sums, size_sums]),
        "모집단 참값": np.concatenate([totals_of("dwelling_class"), totals_of("hh_size")]),
    }
)
comparison["상대 오차"] = (
    (comparison["추정"] - comparison["모집단 참값"]) / comparison["모집단 참값"]
)

check(
    "상위 계층 개체 수",
    "가구 수 계급별 거처 수가 모집단 참값을 0.5% 이내로 재현한다",
    np.all(np.abs(comparison["상대 오차"].to_numpy()[:3]) < 0.005),
)
check(
    "N차원 교차표",
    "가구원 수 계급별 가구 수가 모집단 참값을 0.5% 이내로 재현한다",
    np.all(np.abs(comparison["상대 오차"].to_numpy()[3:]) < 0.005),
)

comparison

,제약 열,추정,모집단 참값,상대 오차
0,hh_class=1,1272.580810,1274.0,-0.001114
1,hh_class=2,237.719845,238.0,-0.001177
2,hh_class=3+,87.563925,88.0,-0.004955
3,size_class=1,408.118392,408.0,0.000290
4,size_class=2,620.497939,621.0,-0.000808
5,size_class=3,460.145998,461.0,-0.001852
6,size_class=4+,521.949947,524.0,-0.003912


두 계급 제약과 총계 사이의 항등식을 확인한다. `4+` 와 `3+` 는 상한이 없는 계급이므로 대표값이 필요하다. 모집단에서 그 계급의 평균값을 구해 쓴다.

In [22]:
# 상한이 없는 계급의 대표값을 모집단에서 구한다.
population_size = gipu.AggregationMapper.descendant_count_to_base(population_tree, "person")
size_class_pop = population_tree.nodes["household"].data["size_class"]
REP_SIZE_4PLUS = float(population_size[size_class_pop == "4+"].mean())

hh_per_dwelling = households.groupby("dwelling_id").size()
REP_HH_3PLUS = float(hh_per_dwelling[dwellings.set_index("dwelling_id")["hh_class"].reindex(
    hh_per_dwelling.index).to_numpy() == "3+"].mean())

size_coefficients = np.array([1.0, 2.0, 3.0, REP_SIZE_4PLUS])
dwelling_coefficients = np.array([1.0, 2.0, REP_HH_3PLUS])

estimated_persons = float(size_sums @ size_coefficients)
estimated_households = float(dwelling_sums @ dwelling_coefficients)

print(f"4+ 가구의 평균 가구원 수 {REP_SIZE_4PLUS:.4f}")
print(f"3+ 거처의 평균 가구 수   {REP_HH_3PLUS:.4f}")
print()
print(f"가구 규모 계급에서 유도한 총 가구원 수 {estimated_persons:9,.0f}  (참값 {len(persons):,})")
print(f"거처 계급에서 유도한 총 가구 수       {estimated_households:9,.0f}  (참값 {len(households):,})")

check(
    "상위 계층 개체 수",
    "거처 계급별 가중합이 총 가구 수 항등식을 1% 이내로 만족한다",
    abs(estimated_households - len(households)) / len(households) < 0.01,
)
check(
    "N차원 교차표",
    "가구 규모 계급별 가중합이 총 가구원 수 항등식을 1% 이내로 만족한다",
    abs(estimated_persons - len(persons)) / len(persons) < 0.01,
)

4+ 가구의 평균 가구원 수 4.4065
3+ 거처의 평균 가구 수   3.0000

가구 규모 계급에서 유도한 총 가구원 수     5,330  (참값 5,342)
거처 계급에서 유도한 총 가구 수           2,011  (참값 2,014)


True

## 9. 주변표의 결측과 구간 추정

공표된 주변표에는 값이 없는 칸이 흔히 있다. 비공개 처리, 반올림 공표, 미공표가 그 원인이다. 여기서는 가구 규모 주변표에서 **2인가구와 3인가구 수가 공표되지 않은** 상황을 가정한다.

선택지는 셋이다.

| 선택 | 결과 |
| --- | --- |
| 결측 셀을 버린다 | 그 범주가 제약 없이 방치되어 조정이 자유롭게 왜곡된다 |
| 점 추정으로 채운다 | 근거 없는 값이 퇴화 구간으로 주입되어 다른 제약과 동등한 강도로 작용한다 |
| **구간으로 연역한다** | 알려진 것만 주장하고, 모르는 부분은 구간의 폭으로 남긴다 |

라이브러리는 셋째를 택한다. 근거는 셀 값들이 만족해야 하는 **균형식** $\sum_k c_k x_k \in [T_{lo}, T_{hi}]$ 이다. 여기서는 두 개를 쓴다.

1. 총 가구 수: $\sum_n H_n = H$ (계수가 모두 1)
2. 총 가구원 수: $\sum_n n H_n = P$ (계수가 계급 대표값)

둘째 균형식의 `4+` 계급 대표값은 소수 첫째 자리까지만 공표되었다고 보고 반올림한다. 그 반올림 오차는 총 가구원 수를 구간으로 주어 흡수한다.

In [23]:
published = dict(zip(SIZE_CLASSES, totals_of("hh_size")))
print("공표되었다고 가정한 실제 값")
for name, value in published.items():
    print(f"  {name:3s} {value:8,.0f}")

# 2인가구와 3인가구가 미공표인 상황
observed_margin = pd.Series(
    {"1": published["1"], "2": np.nan, "3": np.nan, "4+": published["4+"]}
)
cells = gipu.marginal_from_series(observed_margin)

ROUNDED_REP = round(REP_SIZE_4PLUS, 1)  # 공표된 대표값은 소수 첫째 자리까지
person_total = len(persons)

estimator = gipu.MissingMarginEstimator(cells)
estimator.add_balance("households", float(len(households)))
estimator.add_balance(
    "persons",
    (person_total * 0.99, person_total * 1.01),  # 대표값 반올림 오차를 구간으로 흡수
    gipu.balance_from_classes(SIZE_CLASSES, [1.0, 2.0, 3.0, ROUNDED_REP]),
)

print(f"\n결측 셀: {estimator.missing_cells}")
print(f"4+ 대표값 {REP_SIZE_4PLUS:.4f} -> 공표값 {ROUNDED_REP}")
frame = estimator.to_frame()
frame

공표되었다고 가정한 실제 값
  1        408
  2        621
  3        461
  4+       524

결측 셀: ['2', '3']
4+ 대표값 4.4065 -> 공표값 4.4


,cell,status,lower,upper,width
0,1,observed,408.00,408.00,0.00
1,2,missing,564.18,671.02,106.84
2,3,missing,410.98,517.82,106.84
3,4+,observed,524.00,524.00,0.00


연역된 구간이 참값을 포함하는지 확인한다. 포함하지 않는다면 그 구간은 근거로 쓸 수 없다.

In [24]:
intervals = estimator.estimate()

contains_truth = all(
    intervals[name].lower - 1e-6 <= published[name] <= intervals[name].upper + 1e-6
    for name in SIZE_CLASSES
)
check("주변표의 결측", "연역된 구간이 모든 셀의 참값을 포함한다", contains_truth)
check(
    "주변표의 결측",
    "공표된 셀은 퇴화 구간으로, 결측 셀은 폭이 있는 구간으로 남는다",
    intervals["1"].is_scalar and not intervals["2"].is_scalar,
)

pd.DataFrame(
    {
        "계급": SIZE_CLASSES,
        "하한": [intervals[n].lower for n in SIZE_CLASSES],
        "참값": [published[n] for n in SIZE_CLASSES],
        "상한": [intervals[n].upper for n in SIZE_CLASSES],
        "구간 폭": [intervals[n].upper - intervals[n].lower for n in SIZE_CLASSES],
    }
)

,계급,하한,참값,상한,구간 폭
0,1,408.00,408.0,408.00,0.00
1,2,564.18,621.0,671.02,106.84
2,3,410.98,461.0,517.82,106.84
3,4+,524.00,524.0,524.00,0.00


균형식이 하나뿐이면 구간이 얼마나 넓어지는지 대조해 본다. 총 가구 수만 알면 두 결측 셀은 잔차를 나눠 가지는 관계일 뿐이므로, 각 셀은 0부터 잔차 전체까지 어디든 갈 수 있다. 둘째 균형식이 들어와야 비로소 좁혀진다.

In [25]:
only_one = gipu.MissingMarginEstimator(cells)
only_one.add_balance("households", float(len(households)))
loose = only_one.estimate()

widths = pd.DataFrame(
    {
        "계급": SIZE_CLASSES,
        "균형식 1개일 때 폭": [loose[n].upper - loose[n].lower for n in SIZE_CLASSES],
        "균형식 2개일 때 폭": [intervals[n].upper - intervals[n].lower for n in SIZE_CLASSES],
    }
)

check(
    "주변표의 결측",
    "균형식을 추가하면 결측 셀의 구간이 좁아진다",
    all(
        intervals[n].upper - intervals[n].lower <= loose[n].upper - loose[n].lower + 1e-9
        for n in SIZE_CLASSES
    ),
)
widths

,계급,균형식 1개일 때 폭,균형식 2개일 때 폭
0,1,0.0,0.00
1,2,1082.0,106.84
2,3,1082.0,106.84
3,4+,0.0,0.00


모순된 제약표는 반복을 시작하기 전에 걸러진다. 구간을 좁히는 도중 하한이 상한을 넘어서면 주어진 값들이 서로 양립할 수 없다는 뜻이다. 이는 목표 정합성의 사전 검사 역할을 겸한다.

In [26]:
contradictory = gipu.MissingMarginEstimator(
    {"1": 400.0, "2": 600.0, "3": 500.0, "4+": 500.0}
)
contradictory.add_balance("households", 1000.0)  # 실제 합은 2,000

try:
    contradictory.estimate()
    detected = False
except ValueError as error:
    detected = True
    print("사전 검사에서 걸러짐:", error)

check("주변표의 결측", "서로 모순된 제약표를 반복 이전에 탐지한다", detected)

사전 검사에서 걸러짐: 관측값과 균형식이 서로 모순되어 구간을 좁힐 수 없습니다. 하한이 상한을 넘어선 셀: ['1', '2', '3', '4+']


True

### 연역한 구간을 목표로 사용

연역된 구간을 그대로 `hh_size` 제약의 목표로 삼아 다시 조정한다. 결측 셀은 구간 안에서 자유롭게 움직이고, 공표된 셀은 값에 고정된다.

구간은 공짜가 아니다. 결측 셀의 목표가 느슨해지면 그 열이 가중치를 끌어당기지 않으므로, 나머지 제약이 홀로 해를 좁혀야 한다. 아래에서 회차 수가 크게 늘어나는 것이 그 대가이다. 값을 모른다는 사실을 정직하게 반영한 결과이지, 알고리즘의 결함이 아니다.

주의할 점이 하나 더 있다. 셀별 구간만으로는 "네 셀의 합이 총 가구 수와 같다"는 결합 조건이 표현되지 않는다. 각 셀이 자기 구간 안에 있으면서도 합이 총계를 벗어날 수 있기 때문이다. 결합 조건까지 부과하려면 `balance_hierarchy` 로 대범주 위계를 얻어 `CoarseBlock` 을 추가하고, `balance_constraint` 가 만든 총계 제약을 함께 등록한다.

In [27]:
missing_lower = np.array([intervals[n].lower for n in SIZE_CLASSES])
missing_upper = np.array([intervals[n].upper for n in SIZE_CLASSES])

missing_registry = build_registry(missing_lower, missing_upper)
missing_result = gipu.GeneralizedIPUEngine.from_registry(
    attribute.matrix, missing_registry, relative_gap=0.005, max_iterations=2000, eta=1.0
).fit(initial_weights)

missing_sums = missing_result.weighted_sums[attribute.block_slices["hh_size"]]

print(f"종료 사유: {missing_result.termination_reason} ({missing_result.n_iterations} 회차)")
print(f"참값을 알 때의 회차   {result.n_iterations}")
print(f"구간으로 남길 때의 회차 {missing_result.n_iterations}")

check("주변표의 결측", "연역한 구간을 목표로 삼은 반복이 수렴한다", missing_result.converged)
check(
    "주변표의 결측",
    "결측을 구간으로 남기면 수렴에 더 많은 회차가 필요하다",
    missing_result.n_iterations > result.n_iterations,
)
# 결측 셀은 구간 안에 놓여야 하고, 공표된 셀은 허용 한계 안에서 값을 재현해야 한다.
missing_at = [SIZE_CLASSES.index(name) for name in estimator.missing_cells]
observed_at = [SIZE_CLASSES.index(name) for name in estimator.observed_cells]

check(
    "주변표의 결측",
    "결측 셀의 가중합이 연역된 구간 안에 놓인다",
    np.all(missing_sums[missing_at] >= missing_lower[missing_at] - 1e-6)
    and np.all(missing_sums[missing_at] <= missing_upper[missing_at] + 1e-6),
)
check(
    "주변표의 결측",
    "공표된 셀은 퇴화 구간이므로 허용 한계 안에서 공표값을 재현한다",
    np.all(
        np.abs(missing_sums[observed_at] - missing_lower[observed_at])
        / missing_lower[observed_at]
        < 0.005
    ),
)

pd.DataFrame(
    {
        "계급": SIZE_CLASSES,
        "상태": ["결측" if name in estimator.missing_cells else "공표" for name in SIZE_CLASSES],
        "하한": missing_lower,
        "추정": missing_sums,
        "상한": missing_upper,
        "참값": [published[n] for n in SIZE_CLASSES],
    }
)

종료 사유: converged (1073 회차)
참값을 알 때의 회차   84
구간으로 남길 때의 회차 1073


,계급,상태,하한,추정,상한,참값
0,1,공표,408.00,408.389253,408.00,408.0
1,2,결측,564.18,605.901547,671.02,621.0
2,3,결측,410.98,471.423197,517.82,461.0
3,4+,공표,524.00,523.333541,524.00,524.0


In [28]:
# 결합 조건을 부과하는 구성. 균형식의 셀을 하나의 대범주로 묶는다.
size_total_hierarchy = estimator.balance_hierarchy("households", hierarchy_name="hh_size_total")
size_total_constraint = estimator.balance_constraint(
    "households", "hh_size_total", "household"
)

print("대범주 위계:", size_total_hierarchy.mapping)
print(f"총계 제약: [{size_total_constraint.lower[0]:,.0f}, {size_total_constraint.upper[0]:,.0f}]")

check(
    "주변표의 결측",
    "균형식으로부터 결합 조건을 부과할 대범주 위계와 총계 제약을 얻는다",
    size_total_hierarchy.coarse_categories == ["households"]
    and size_total_constraint.n_columns == 1,
)

대범주 위계: {'households': ['1', '2', '3', '4+']}
총계 제약: [2,014, 2,014]


True

## 10. 표본 영의 처리

구조적 영과 표본 영은 둘 다 가중합이 0인 제약 열이지만, 판정 근거와 처리 방식이 다르다.

| 구분 | 판정 근거 | 모집단에서의 참값 | 처리 |
| --- | --- | --- | --- |
| 구조적 영 | 도메인 규칙 | 0 | 색인 원천 제외 |
| 표본 영 | 표본 관측 결과 | 양수, 값 미상 | $\varepsilon$ 평활 |

혼동하면 양방향의 오류가 생긴다. 구조적 영을 표본 영으로 오인하면 불가능한 개체에 가중치가 배분되고, 표본 영을 구조적 영으로 오인하면 실재하는 범주가 영구히 0으로 고정된다.

표본 영을 만들기 위해 **단독가구 거처만 뽑은** 표본을 구성한다. 이 표본에는 다가구주택이 한 호도 없으므로, `hh_class=2` 와 `hh_class=3+` 제약 열의 가중합이 0이 된다. 모집단에는 존재하므로 목표는 양수이다.

In [29]:
single_ids = sample_dwellings.loc[sample_dwellings["hh_class"] == "1", "dwelling_id"]
tiny_dwellings = sample_dwellings[sample_dwellings["dwelling_id"].isin(single_ids)].reset_index(drop=True)
tiny_households = sample_households[
    sample_households["dwelling_id"].isin(set(single_ids))
].reset_index(drop=True)
tiny_persons = sample_persons[
    sample_persons["hh_id"].isin(set(tiny_households["hh_id"]))
].reset_index(drop=True)

tiny_tree = build_tree(tiny_dwellings, tiny_households, tiny_persons)
tiny_attribute = build_attribute_matrix(tiny_tree)
tiny_weights = np.ones(tiny_attribute.n_units)

zero_columns = gipu.ZeroCellResolver.detect(
    tiny_attribute.matrix, tiny_weights, registry.bounds()[0]
)
print(f"단독가구 거처만 뽑은 표본: 거처 {len(tiny_dwellings):,}호, 가구 {len(tiny_households):,}호")
print("표본 영으로 판정된 제약 열:")
for column in zero_columns:
    print(f"  [{column}] {tiny_attribute.column_names[column]}")

check(
    "표본 영",
    "표본에 없으나 목표가 양수인 제약 열을 표본 영으로 탐지한다",
    len(zero_columns) > 0
    and all("hh_class" in tiny_attribute.column_names[c] for c in zero_columns),
)

단독가구 거처만 뽑은 표본: 거처 184호, 가구 184호
표본 영으로 판정된 제약 열:
  [16] hh_class=2
  [17] hh_class=3+


True

표본 영을 그대로 두면 갱신 비율의 분모가 0이 되어 비율이 정의되지 않는다. 라이브러리는 이런 열의 비율을 1로 두므로, 가중치는 **영원히 움직이지 않는다**. 그 범주는 조정에서 배제된 채 남는다.

$\varepsilon$ 평활은 해당 열에만 미세 의사 빈도를 주입하여 비율을 정의 가능하게 만든다. 조밀 변환 없이 좌표 형식의 증분 행렬을 더하므로 희소성이 유지된다.

In [30]:
before_ratios = compute_update_ratios(
    tiny_attribute.matrix.T.dot(tiny_weights), *registry.bounds()
)

smoothed, resolved = gipu.ZeroCellResolver.resolve(
    tiny_attribute.matrix, tiny_weights, registry.bounds()[0]
)
after_ratios = compute_update_ratios(smoothed.T.dot(tiny_weights), *registry.bounds())
remaining = gipu.ZeroCellResolver.detect(smoothed, tiny_weights, registry.bounds()[0])

check(
    "표본 영",
    "평활 이전에는 표본 영 제약 열의 갱신 비율이 1이어서 가중치가 움직이지 않는다",
    np.all(before_ratios[zero_columns] == 1.0),
)
check("표본 영", "평활 이후 표본 영으로 판정되는 제약 열이 없다", len(remaining) == 0)
check(
    "표본 영",
    "평활은 대상 열에만 적용되어 희소성이 유지된다",
    smoothed.nnz - tiny_attribute.nnz == len(resolved) * tiny_attribute.n_units,
)

pd.DataFrame(
    {
        "제약 열": [tiny_attribute.column_names[c] for c in zero_columns],
        "평활 전 비율": before_ratios[zero_columns],
        "평활 후 비율": after_ratios[zero_columns],
    }
)

,제약 열,평활 전 비율,평활 후 비율
0,hh_class=2,1.0,129347.826087
1,hh_class=3+,1.0,47826.086957


평활 후의 비율이 매우 큰 값이라는 점에 주목한다. $\varepsilon$ 평활은 나눗셈이 정의되게 만들 뿐, 표본에 없는 개체를 만들어내지는 못한다. 이 상태에서 반복을 돌리면 그 목표를 맞추기 위해 전체 가중치가 비현실적으로 부풀려진다.

따라서 표본 영이 다수 탐지되면 평활에 기대지 말고, 표본을 보강하거나 그 제약을 더 굵은 범주로 묶는 편이 옳다. 진단이 먼저이고 평활은 그 다음이다.

## 11. 완화 계수의 영향

완화 계수 $\eta$ 를 낮추면 한 회차의 이동 폭이 줄어든다. 진동이 억제되는 대신 수렴에 더 많은 회차가 필요하다. 과잉 제약이 의심될 때 가장 먼저 시도할 대응이다.

In [31]:
rows = []
for eta in [1.0, 0.7, 0.4, 0.2]:
    trial = gipu.GeneralizedIPUEngine.from_registry(
        attribute.matrix, registry, relative_gap=0.005, max_iterations=800, eta=eta
    ).fit(initial_weights)
    rows.append(
        {
            "eta": eta,
            "종료 사유": trial.termination_reason,
            "회차": trial.n_iterations,
            "최대 상대 격차": trial.report.max_relative_gap,
            "유효 표본 수": gipu.WeightDistributionAnalyzer.analyze(
                trial.weights
            ).effective_sample_size,
        }
    )

eta_frame = pd.DataFrame(rows)
converged_trials = eta_frame[eta_frame["종료 사유"] == "converged"]

check(
    "구간 목표",
    "완화 계수를 낮추면 수렴에 필요한 회차가 늘어난다",
    converged_trials["회차"].is_monotonic_increasing
    if len(converged_trials) > 1
    else True,
)
eta_frame

,eta,종료 사유,회차,최대 상대 격차,유효 표본 수
0,1.0,converged,84,0.004955,294.453462
1,0.7,converged,120,0.004980,294.437274
2,0.4,converged,211,0.004981,294.420726
3,0.2,converged,422,0.004996,294.410692


## 12. 실현 불가능한 제약

서로 양립할 수 없는 목표를 주면 반복은 수렴하지 않는다. 아래는 연령 대범주 목표를 소범주 목표의 두 배로 부풀린 경우이다. 두 제약이 가중치를 반대 방향으로 끌어당기므로, 알고리즘은 최대 반복에 도달한다.

이때 보고서 상단을 보면 어느 제약이 상충하는지 드러난다. 미수렴은 알고리즘의 실패가 아니라 제약 집합의 성질이다.

In [32]:
infeasible = gipu.ConstraintRegistry()
infeasible.register(
    gipu.BoundChecker.from_arrays(
        "person_age_sex", "person", names_of("person_age_sex"), totals_of("person_age_sex")
    )
)
infeasible.register(
    gipu.BoundChecker.from_arrays(
        "person_age", "person", names_of("person_age"),
        totals_of("person_age") * 2.0,  # 소범주 합의 두 배: 성립할 수 없다
    )
)

conflict_matrix = attribute.matrix[
    :, list(range(attribute.block_slices["person_age_sex"].start,
                  attribute.block_slices["person_age"].stop))
]
conflict_result = gipu.GeneralizedIPUEngine.from_registry(
    conflict_matrix, infeasible, relative_gap=0.005, max_iterations=60
).fit(initial_weights)

print(f"종료 사유: {conflict_result.termination_reason}")
print(f"진단 소견: {conflict_result.tracker.diagnose()}")
print(f"미충족 제약 열: {conflict_result.report.n_violations} / {conflict_matrix.shape[1]}")

check(
    "구간 목표",
    "실현 불가능한 제약에서 수렴이 아닌 최대 반복 도달로 종료한다",
    conflict_result.termination_reason == "iteration_limit_reached",
)

gipu.FitReportGenerator.generate_report(
    conflict_result.weighted_sums,
    *infeasible.bounds(),
    column_names=infeasible.column_names,
    column_owners=infeasible.column_owners,
    relative_gap=0.005,
).head(6)

종료 사유: iteration_limit_reached
진단 소견: 감소 중
미충족 제약 열: 9 / 9


,constraint,column,lower,upper,estimated,bound_violation,relative_gap,within_bounds,satisfied
0,person_age_sex,person_age_sex::age_group=senior|sex=F,578.0,578.0,867.015004,289.015004,0.500026,False,False
1,person_age_sex,person_age_sex::age_group=child|sex=M,567.0,567.0,850.513422,283.513422,0.500024,False,False
2,person_age_sex,person_age_sex::age_group=adult|sex=F,1591.0,1591.0,2386.523271,795.523271,0.500015,False,False
3,person_age_sex,person_age_sex::age_group=adult|sex=M,1558.0,1558.0,2336.977412,778.977412,0.499986,False,False
4,person_age_sex,person_age_sex::age_group=senior|sex=M,531.0,531.0,796.485296,265.485296,0.499972,False,False
5,person_age_sex,person_age_sex::age_group=child|sex=F,517.0,517.0,775.485595,258.485595,0.499972,False,False


## 13. 결과 내보내기

최종 가중치를 원본 자료에 결합한다. 가중치 벡터의 성분 순서는 기본 계층 자료의 행 순서와 같지만, 순서에 의존하면 어긋났을 때 오류 없이 잘못된 값이 부여된다. `AttributeMatrix.base_keys` 를 넘겨 **주키를 기준으로 결합**하는 편이 안전하다.

In [33]:
weighted_households = gipu.DatasetExporter.attach_weights(
    sample_households, result.weights, keys=attribute.base_keys, key_column="hh_id"
)
weighted_households.head()

,hh_id,dwelling_id,car,ipu_weight
0,h00006,d00005,1.0,3.633397
1,h00007,d00005,1.0,5.245212
2,h00012,d00009,0.0,6.591589
3,h00024,d00020,1.0,6.798502
4,h00025,d00020,1.0,5.637019


가구에 부여된 가중치는 하위 계층인 개인 자료로 전파할 수 있다. 같은 가구에 속한 개인은 그 가구의 가중치를 공유한다.

In [34]:
weighted_persons = gipu.DatasetExporter.propagate_to_level(
    sample_tree, "person", result.weights
)

estimated_population = float(weighted_persons["ipu_weight"].sum())
print(f"가중 개인 수 추정 {estimated_population:,.0f} 명")
print(f"모집단 실제 개인 수 {len(persons):,} 명")

check(
    "N계층 구조",
    "기본 계층의 가중치를 하위 계층으로 전파하여 총인구를 1% 이내로 재현한다",
    abs(estimated_population - len(persons)) / len(persons) < 0.01,
)

weighted_persons.head()

가중 개인 수 추정 5,345 명
모집단 실제 개인 수 5,342 명


,person_id,hh_id,age_group,sex,license,ipu_weight
0,p000018,h00006,senior,M,0,3.633397
1,p000019,h00006,adult,F,1,3.633397
2,p000020,h00006,senior,M,0,3.633397
3,p000021,h00007,child,F,0,5.245212
4,p000022,h00007,child,F,0,5.245212


개체 수준의 합성 자료가 필요하면 가중치를 정수로 바꾼다. `integerize` 는 소수부를 성공 확률로 삼는 확률적 반올림을 사용하므로 총합의 기댓값이 보존된다.

In [35]:
integer_weights = gipu.DatasetExporter.integerize(result.weights, random_state=0)

print(f"실수 가중치 총합 {result.weights.sum():,.1f}")
print(f"정수 가중치 총합 {integer_weights.sum():,}")
print(f"차이 {abs(integer_weights.sum() - result.weights.sum()):,.1f}")

실수 가중치 총합 2,010.7
정수 가중치 총합 1,990
차이 20.7


파일로 저장할 때는 `export` 또는 `export_with_weights` 를 사용한다. 확장자로 형식을 판정하며 CSV와 Parquet을 지원한다.

```python
gipu.DatasetExporter.export(weighted_households, "synthetic_households.csv")
gipu.DatasetExporter.export(weighted_persons, "synthetic_persons.parquet")
```

## 14. 검증 요약

각 절에서 기록한 검증을 문제별로 모은다. `check()` 는 조건이 거짓이면 그 자리에서 예외를 발생시키므로, 이 표가 출력되었다는 것은 모든 항목이 통과했다는 뜻이다.

In [36]:
checks = pd.DataFrame(CHECKS)

print(f"검증 {len(checks)}건, 통과 {int(checks['통과'].sum())}건")
print()
print(checks.groupby("문제", sort=False)["통과"].agg(["size", "sum"]).rename(
    columns={"size": "검증 수", "sum": "통과"}
).to_string())

assert checks["통과"].all(), "통과하지 못한 검증이 있다"
checks

검증 33건, 통과 33건

            검증 수  통과
문제                  
N계층 구조         4   4
구조적 영          2   2
범주 위계          1   1
상위 계층 개체 수     5   5
N차원 교차표        3   3
구간 목표          5   5
주변표의 결측        9   9
표본 영           4   4


,문제,검증 내용,통과
0,N계층 구조,표본은 거처 단위로 추출되어 다가구주택의 가구가 통째로 포함된다,True
1,N계층 구조,참조 무결성 위반을 반복 이전에 예외로 걸러낸다,True
2,구조적 영,규칙이 지정한 조합 1개만 유효 마스크에서 거짓이 된다,True
3,구조적 영,구조적 영 제약 열은 속성 행렬에 색인이 할당되지 않는다,True
4,범주 위계,대범주 블록이 소범주 블록과 매핑 행렬의 곱과 정확히 일치한다,True
5,상위 계층 개체 수,방송한 열의 가중합은 거처 수가 아니라 가구 수가 된다,True
6,상위 계층 개체 수,분수 귀속 열의 가중합은 표본 거처 수와 일치한다,True
7,상위 계층 개체 수,가구 수 계급별 거처 수가 표본의 실제 분포와 일치한다,True
8,N계층 구조,계층에서 유도한 가구원 수가 개인 자료를 직접 센 값과 일치한다,True
9,N차원 교차표,가구원 수 계급별 가구 수가 표본의 실제 분포와 일치한다,True


## 정리

이 노트북에서 다룬 절차는 다음과 같다.

1. 계층별 자료를 `HierarchyTree` 로 묶고 무결성을 검증한다. 하위 계층 개체 수처럼 원본에 없는 값은 계층 구조에서 유도한다.
2. 불가능한 범주 조합을 `LogicalRuleParser` 로 걸러 유효 마스크를 만든다.
3. 주변표에 결측이 있으면 `MissingMarginEstimator` 로 목표 구간을 연역한다. 이 단계에서 제약표 사이의 모순도 함께 걸러진다.
4. `UnifiedSparseMatrixBuilder` 로 블록을 결합하여 속성 행렬을 구축한다. 구조적 영은 이 단계에서 색인 자체가 제외되고, 상위 계층 개체 수는 `ShareBlock` 의 분수 귀속으로 계상된다.
5. 목표를 구간으로 정규화하여 `ConstraintRegistry` 에 등록한다.
6. `GeneralizedIPUEngine` 으로 반복을 수행하고 종료 사유를 확인한다.
7. 적합도와 가중치 분포를 함께 진단한다. 표본 영이 다수 탐지되면 평활보다 표본 보강이나 범주 병합을 먼저 검토한다.
8. 주키를 기준으로 가중치를 결합하여 내보낸다.

미구현 항목은 [`docs/03_잔여_과제.md`](../docs/03_잔여_과제.md)에 정리되어 있다.